# Minimal reproduction: a slot with `slot_uri: rdf:type` collides with a node's own class-typing triple

Prepared as supporting evidence for
[linkml/linkml#3931](https://github.com/linkml/linkml/issues/3931) --
following up on
[nfdi-de/dcat-ap-plus#110](https://github.com/nfdi-de/dcat-ap-plus/issues/110),
where the same bug was first found (via dcat-ap-plus's own `ClassifierMixin`,
see `examples/issues_for_dcat-ap-plus/classifiermixin-rdf-type-collision.ipynb`
next to this repo's own copy of that notebook). The maintainer there
(StroemPhi) confirmed the bug independently, called it "a pure SHACL
thing," and suggested the SHACL generator should emit a `sh:or` union in
such extra-typing cases, where `rdf:type` is used as a `slot_uri` -- see
the last cell of this notebook. A LinkML maintainer (matentzn) then asked
for it to be filed on the LinkML tracker directly, since it's a
`ShaclGenerator` bug, not a dcat-ap-plus one.

**This reproduction has zero dependency on dcat-ap-plus** (or on
Health-DCAT-AP-plus/ResHealth-DCAT-AP), unlike the earlier notebook --
deliberately, to demonstrate this is a genuine `linkml.generators.shaclgen`
bug, not something specific to dcat-ap-plus's own schema design. The
schema used here (`minimal-rdf-type-collision.yaml`, next to this file) is
two classes and one slot, nothing else.

**Deliberately atomic**, matching the same standard as the earlier
notebook: exactly one triple, a bare class-typing assertion, nothing
else -- no extra slots, no literals, so there's no risk of entangling
this with any other, separately-tracked issue.

In [1]:
from pathlib import Path
import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

SCHEMA = Path("minimal-rdf-type-collision.yaml")
assert SCHEMA.exists()

## 1. Generate SHACL from the minimal schema

Two classes (`Thing`, `DefinedTerm`), one slot (`rdf_type`, `slot_uri:
rdf:type`, `range: DefinedTerm`). No imports beyond `linkml:types`.

In [2]:
shapes_ttl = ShaclGenerator(str(SCHEMA)).serialize()
print(shapes_ttl)

@prefix ex: <https://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:Thing a sh:NodeShape ;
    rdfs:comment "Any class that also wants to record an *additional*, explicit rdf:type value via a real slot (a common pattern -- e.g. dcat-ap-plus's ClassifierMixin, PROV-O's own \"extra typing\" convention) collides with its own class-assertion triple this way." ;
    sh:closed true ;
    sh:ignoredProperties ( rdf:type ) ;
    sh:property [ sh:class ex:DefinedTerm ;
            sh:description "An extra, explicit rdf:type value, distinct from the node's own class-typing triple -- but SHACL generation can't tell them apart, since both use the same predicate." ;
            sh:maxCount 1 ;
            sh:nodeKind sh:BlankNodeOrIRI ;
            sh:order 0 ;
            sh:path rdf:type ] ;
    sh:targetClass ex:T

Note the `Thing` shape's `sh:property` for `rdf:type`: `sh:class
ex:DefinedTerm`. That constraint is meant for values of the `rdf_type`
*slot* -- but `rdf:type` is also the predicate `Thing`'s own `sh:targetClass
ex:Thing` match depends on, and the generated shape has no way to tell the
two apart.

## 2. The one-triple example -- a bare class assertion, nothing else

In [3]:
data_ttl = """
@prefix ex: <https://example.org/> .

<https://example.org/x> a ex:Thing .
"""
data_graph = Graph()
data_graph.parse(data=data_ttl, format="turtle")
print(data_graph.serialize(format="turtle"))

@prefix ex: <https://example.org/> .

ex:x a ex:Thing .




No `rdf_type` slot value was ever set -- this node has exactly one triple,
its own class assertion.

## 3. Validate

In [4]:
shapes_graph = Graph()
shapes_graph.parse(data=shapes_ttl, format="turtle")

conforms, results_graph, results_text = pyshacl.validate(
    data_graph,
    shacl_graph=shapes_graph,
    data_graph_format="turtle",
    inference="none",
    advanced=True,
)
print(f"conforms = {conforms}")
print(results_text)

conforms = False
Validation Report
Conforms: False
Results (1):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class ex:DefinedTerm ; sh:description Literal("An extra, explicit rdf:type value, distinct from the node's own class-typing triple -- but SHACL generation can't tell them apart, since both use the same predicate.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:BlankNodeOrIRI ; sh:order Literal("0", datatype=xsd:integer) ; sh:path rdf:type ]
	Focus Node: ex:x
	Value Node: ex:Thing
	Result Path: rdf:type
	Message: Value does not have class ex:DefinedTerm



## What happened

`conforms = False`. The value node in the violation is `ex:Thing` itself
-- the node's *own class assertion*, not a value anyone set via the
`rdf_type` slot. Same failure mode as the earlier dcat-ap-plus-specific
reproduction, now confirmed with a genuinely minimal schema that has
nothing to do with dcat-ap-plus, PROV-O, or health data -- just a slot
with `slot_uri: rdf:type` on a class that also has its own `class_uri`.

**Root cause**: `rdf:type` serves two distinct roles in RDF -- (1) the
predicate every node uses to assert its own class membership (what makes
`sh:targetClass` match it), and (2) an ordinary predicate a schema is free
to reuse for a real slot (`slot_uri: rdf:type`). LinkML's `ShaclGenerator`
generates one property shape for role (2), constrained by the slot's own
range -- but SHACL applies that shape to *every* triple with predicate
`rdf:type` on a matching node, including the ones from role (1), which it
has no way to distinguish.

**Suggested fix direction**, from dcat-ap-plus maintainer StroemPhi's own
review of the earlier reproduction: have the SHACL generator emit a
`sh:or` union in this specific case (a slot whose `slot_uri` is
`rdf:type`, on a class with its own `class_uri`) -- one branch matching
the slot's own range constraint (`sh:class DefinedTerm`), the other
matching the node's own declared class(es) (e.g. `sh:hasValue ex:Thing`,
or `sh:class` against the class hierarchy's own `class_uri` values) -- so
a value node satisfying *either* role no longer triggers a spurious
violation.